https://gist.github.com/waveletdeboshir/8bf52f04bf78018194f25b2390c08309

In [1]:
import os
from pathlib import Path

In [2]:
from transformers import pipeline
import torch

2026-04-22 15:24:46.981759: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-22 15:24:47.090563: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-22 15:24:48.820533: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/arman/it/AI_work/.venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8

In [3]:
torch.cuda.empty_cache()
print(torch.cuda.memory_summary(device=None, abbreviated=False))

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |

In [4]:
audio_path = Path("dataset/kish_durak_i_molniya.mp3")
assert audio_path.exists(), f"File not found: {audio_path}"

audio_path

PosixPath('dataset/kish_durak_i_molniya.mp3')

In [5]:
whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3",
    use_fast=False,
)

Device set to use cuda:0


In [6]:
torch.cuda.empty_cache()
print(torch.cuda.memory_summary(device=None, abbreviated=False))

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   2944 MiB |   2944 MiB |   2944 MiB |      0 B   |
|       from large pool |   2941 MiB |   2941 MiB |   2941 MiB |      0 B   |
|       from small pool |      3 MiB |      3 MiB |      3 MiB |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |   2944 MiB |   2944 MiB |   2944 MiB |      0 B   |
|       from large pool |   2941 MiB |   2941 MiB |   2941 MiB |

whisper_result = whisper_pipe(
    str(audio_path),
    chunk_length_s=30,
    batch_size=8,
    return_timestamps=True,
    generate_kwargs={"language": "russian", "task": "transcribe"},
)

In [8]:
whisper_result = whisper_pipe(
    str(audio_path),
    return_timestamps=True,
    generate_kwargs={"language": "russian", "task": "transcribe"},
)


In [9]:
whisper_text = whisper_result["text"].strip()

print("Whisper ASR:\n")
print(whisper_text)

whisper_result

Whisper ASR:

Грохочет гром Всё поймать стремится, ты вовремя любишь Весь сельский люд смотреть на это выходил Как на холме безумец бегал и щудил Он видно в ссоре с головою, видно сам в себе он враг Надо ж выдумать такое, о, дурак То парик лесу мчится, то в поле, то к ручью, Кто поймать стремится, мол, иди. Утром по сельской дороге медленно шел ночной герой, Весь лохматый и седой, и улыбался. Редактор субтитров А.Семкин


{'text': ' Грохочет гром Всё поймать стремится, ты вовремя любишь Весь сельский люд смотреть на это выходил Как на холме безумец бегал и щудил Он видно в ссоре с головою, видно сам в себе он враг Надо ж выдумать такое, о, дурак То парик лесу мчится, то в поле, то к ручью, Кто поймать стремится, мол, иди. Утром по сельской дороге медленно шел ночной герой, Весь лохматый и седой, и улыбался. Редактор субтитров А.Семкин',
 'chunks': [{'timestamp': (0.0, 4.76), 'text': ' Грохочет гром'},
  {'timestamp': (30.0, 36.0),
   'text': ' Всё поймать стремится, ты вовремя любишь'},
  {'timestamp': (36.0, 41.16),
   'text': ' Весь сельский люд смотреть на это выходил'},
  {'timestamp': (41.16, 46.48), 'text': ' Как на холме безумец бегал и щудил'},
  {'timestamp': (46.48, 51.9),
   'text': ' Он видно в ссоре с головою, видно сам в себе он враг'},
  {'timestamp': (51.9, 56.1), 'text': ' Надо ж выдумать такое, о, дурак'},
  {'timestamp': (56.1, 61.3),
   'text': ' То парик лесу мчится, то в поле, то к